# 05B — Potential Improvements

**Rubric target:** Potential Improvements — "Propose strategies to enhance the model further."

Each improvement is tied to a specific bias or limitation identified in the EDA,
explains what signal it would add, and references the relevant syllabus topic.

---
## 1. Sequential Purchase Modeling with GRU4Rec

**Syllabus coverage:** Week 7 — RNNs, LSTMs, GRUs, sequence modeling

**Problem addressed:** Our current pipeline treats each customer's purchase history
as a *set* — it knows what they bought, but not the *order* in which they bought it.
A customer who bought a summer dress → sandals → sunglasses is on a different
trajectory than one who bought sunglasses → a winter coat → boots, even if both
have bought the same items overall.

**Proposed approach:** GRU4Rec (Hidasi et al., 2015) is a session-based recommendation
model that uses a **Gated Recurrent Unit** (GRU) — a type of recurrent neural network —
to model the sequence of items a user has interacted with. At each timestep, the GRU
updates a hidden state that captures the evolving context of the session:

- **Update gate:** Controls how much of the previous hidden state to retain
- **Reset gate:** Controls how much of the previous state to forget when computing new candidates
- **Hidden state:** A dense vector summarizing the purchase sequence so far

The model takes a sequence of purchased item embeddings as input and predicts the
probability of each item being the next purchase. This directly captures temporal
patterns: seasonal transitions (summer → fall wardrobe), complementary sequences
(blazer on Monday → trousers on Thursday), and evolving taste over time.

**What signal it would add:** Our EDA showed that 80% of multi-item baskets span
departments and that sequential complement patterns are strong (19% of shorts
buyers purchase swimwear within 7 days). A GRU would learn these temporal
transitions directly from the data rather than relying on static co-purchase
counts.

**Why we didn't build it:** GRU4Rec requires a fundamentally different training
paradigm (session-based, not candidate-pair-based) and significant hyperparameter
tuning. It's a separate architecture, not a feature that plugs into LightGBM.
However, GRU4Rec scores could be used as an additional candidate source or as
a feature in the ranker (the GRU's predicted probability for each candidate item).

**Bias addressed:** Seasonal confounding (Bias 3) — the GRU would implicitly
learn seasonal purchase sequences rather than treating all historical purchases
as equally informative regardless of time of year.

---
## 2. Complement Classifier for Candidate Generation

**Problem addressed:** Our co-purchase candidate source uses raw co-occurrence
counts to recommend items bought alongside a customer's recent purchases.
This conflates substitutes (another black t-shirt) with complements (trousers
to go with the blazer). A dedicated classifier could distinguish between them.

**Proposed approach:** Train a binary classifier on item pairs, where:
- **Complement pairs (label=1):** Items frequently co-purchased in the same basket
  but from *different* product categories (blazer + trousers, bikini top + swimwear bottom)
- **Substitute pairs (label=0):** Items from the *same* product category purchased by
  overlapping customers but rarely in the same basket (two different styles of jeans)

Features for the classifier would include:
- Image embedding cosine similarity (substitutes should look alike; complements shouldn't)
- Text embedding similarity (similar descriptions → substitutes)
- Same/different department, product type, color group
- Co-purchase lift (from our EDA)
- Price ratio between the two items

**What signal it would add:** The EDA's complement analysis revealed strong
directional asymmetries: P(trousers | bought blazer) ≠ P(blazer | bought trousers).
A complement classifier would generate directed recommendations — "you bought A,
you probably need B" — rather than undirected co-occurrence.

**Integration:** Replace the co-purchase candidate source with complement-classified
candidates. For each of a customer's recent purchases, retrieve the top complement
items (not substitutes) as candidates. This would shift the candidate pool toward
discovery (new items that complete an outfit) rather than redundancy (more of what
they already own).

**Bias addressed:** Popularity bias (Bias 2) — complement recommendations would
naturally surface niche items that pair well with popular ones, increasing
long-tail exposure.

---
## 3. Enhanced Image Embeddings with Fine-Tuning or CLIP

**Problem addressed:** Our ResNet-50 embeddings are pretrained on ImageNet (natural
images of animals, objects, scenes) and used without fine-tuning. Fashion images
have a different visual vocabulary — fabric texture, silhouette, pattern, drape —
that ImageNet features may not capture well.

**Proposed approaches:**

*Option A: Fine-tune on fashion data.* Use the co-purchase pairs from the EDA as
weak supervision — items bought together should have similar embeddings (contrastive
learning). This adapts the visual features to the fashion domain.

*Option B: Use CLIP (Contrastive Language-Image Pretraining).* CLIP jointly embeds
images and text in the same vector space. A product's image embedding and its
text description would be directly comparable, enabling cross-modal retrieval
("find items that look like what this description says"). This could replace
both our separate image and text embeddings with a single unified representation.

**What signal it would add:** Our ablation showed that current embeddings contribute
+4.28% MAP@12 improvement. Fashion-adapted embeddings would likely increase this
by capturing domain-specific visual similarity (e.g., recognizing that a slim-fit
navy blazer is more similar to a slim-fit charcoal blazer than to an oversized
denim jacket, even though ImageNet features might rate them equally).

**Bias addressed:** Implicit feedback bias (Bias 1) — better visual representations
would improve item-item similarity, helping the system recommend visually
appealing items even for customers with minimal purchase history.

---
## 4. Graph Neural Networks on the User-Item Bipartite Graph

**Problem addressed:** ALS captures global latent patterns but loses the graph
structure of who-bought-what. Multi-hop relationships ("customers who bought
what your neighbors bought") are lost in matrix factorization.

**Proposed approach:** Construct a bipartite graph where users and items are nodes,
and purchases are edges (weighted by recency). Apply a Graph Neural Network
(e.g., LightGCN or PinSage) to learn node embeddings through message passing:
each node aggregates information from its neighbors over multiple hops.

After K layers of message passing, a user's embedding reflects not just their
own purchases but the purchases of similar users (2-hop neighbors), and the
users similar to *those* users (3-hop), and so on.

**What signal it would add:** Higher-order collaborative signals. ALS with
rank=256 captured first-order patterns effectively (MAP@12 = 0.0154), but
GNNs could capture the multi-hop relational structure that factorization
compresses away.

**Bias addressed:** Sparse user histories (Bias 4) — GNNs propagate information
from dense neighborhoods to sparse ones, partially alleviating cold-start by
borrowing signal from connected users.

---
## 5. Online Learning / Incremental Updates

**Problem addressed:** Our model is trained once on historical data and never
updated. In production, new purchases happen continuously, and a customer's
preferences may change within hours (they buy a gift, start a new fitness
routine, change style).

**Proposed approach:** Implement an incremental update mechanism:
- Update user embeddings (ALS factors) as new purchases arrive, without
  retraining the full model
- Maintain a real-time popularity index that refreshes hourly
- Re-rank candidates periodically using the latest interaction features

**What signal it would add:** Real-time responsiveness. If a customer buys
a blazer at 10 AM, their afternoon recommendations should include matching
trousers — not wait for a daily batch retrain.

**Bias addressed:** No browse/negative signal (Bias 6) — in a production
system with online learning, we could incorporate click-through data,
cart additions, and page views as they happen, dramatically enriching
the implicit feedback signal beyond purchases alone.

---
## 6. Candidate Recall Optimization

**Problem addressed:** Our candidate recall is 9.3% — meaning 90.7% of items
customers actually purchased were not in the candidate pool. This is the
binding constraint on our entire pipeline; no ranker can recommend items
outside the candidate set.

**Proposed approaches:**
- **Item2Vec:** Train item embeddings directly from purchase sequences
  (analogous to Word2Vec on sentences), then retrieve nearest-neighbor
  items in embedding space. The 1st place Kaggle solution found this
  highly effective.
- **Product-code variant retrieval:** For each item in a customer's history,
  include all color variants of that product code as candidates. Our EDA
  showed that product codes average 2.2 variants — this is essentially
  free recall.
- **Larger candidate pools:** Our current ~74 candidates per user could
  be expanded to 100-150 with moderate compute cost, directly improving
  the recall ceiling.

**Bias addressed:** Catalog selection bias (Bias 7) — more diverse candidate
sources would surface items from underrepresented categories and reduce
the popularity feedback loop.

---
## Summary: Improvements Roadmap

| Improvement | Signal Added | Bias Addressed | Effort | Expected Impact |
|-------------|-------------|----------------|--------|----------------|
| GRU4Rec (RNN) | Temporal purchase sequences | Seasonal confounding | High | Medium-High |
| Complement classifier | Directional outfit completion | Popularity bias | Medium | Medium |
| CLIP / fine-tuned CNN | Fashion-specific visual similarity | Implicit feedback | Medium | Low-Medium |
| Graph Neural Networks | Multi-hop collaborative patterns | Sparse histories | High | Medium |
| Online learning | Real-time responsiveness | No browse data | High | High |
| Candidate recall optimization | More diverse retrieval | Catalog selection | Low | High |

The highest-ROI improvement would be **candidate recall optimization** — it's
low effort (adding product-code variants and expanding pool sizes) with the
highest expected impact (directly raises the performance ceiling for the entire
pipeline). The most technically interesting would be the **complement classifier**,
which builds directly on the co-purchase analysis from our EDA.